# Lesson 08 — Working with dates

In this lesson:
- Meet two new tables: **`parks`** and **`attractions`**
- Learn the `DATE` type
- Pull out parts of a date with `EXTRACT`
- Compute differences between dates with `DATE_DIFF`
- Compare dates against today using `CURRENT_DATE`

Time: 25 minutes.

Dates show up in almost every real-world database. Once you can work with them, a huge number of common questions become easy.

## Setup

In [ ]:
# Sign in to Google and connect to the Disney dataset.
# Run this once per Colab session.

from google.colab import auth
auth.authenticate_user()

from google.cloud.bigquery import magics
magics.context.project = "sql-with-disney"

print("✓ Ready! Run the query cells below.")

## Meet the `parks` table

12 rows — every Disney theme park worldwide:

| Column | Type | Meaning |
|--------|------|---------|
| `park_name` | text | Name of the park |
| `location` | text | City, country |
| `opened_date` | **DATE** | The day the park opened |
| `size_acres` | number | Total acreage |

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.parks
ORDER BY opened_date

Notice the `opened_date` column displays as `1955-07-17`, `1971-10-01`, etc. That's a `DATE` value — year, month, day. SQL knows it's a date (not just text), so you can do date math on it.

## Date literals

When you want to write a specific date in SQL, use the `DATE 'YYYY-MM-DD'` form:

```sql
DATE '1955-07-17'
```

Try filtering by a specific date:

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.parks
WHERE opened_date >= DATE '2000-01-01'
ORDER BY opened_date

All parks opened in the 21st century. Note that date comparisons (`<`, `>=`, etc.) work the same way as numeric comparisons.

## `EXTRACT` — pull out parts of a date

`EXTRACT` lets you grab the year, month, day (or other parts) out of a date.

The shape: `EXTRACT(YEAR FROM date_column)`. Replace `YEAR` with `MONTH`, `DAY`, etc.

How many parks opened each year? (Group by extracted year.)

In [ ]:
%%bigquery
SELECT
  EXTRACT(YEAR FROM opened_date) AS opened_year,
  COUNT(*) AS parks_opened
FROM disney_lessons.parks
GROUP BY opened_year
ORDER BY opened_year

Filter to a decade — parks opened in the 1990s:

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.parks
WHERE EXTRACT(YEAR FROM opened_date) BETWEEN 1990 AND 1999

## `DATE_DIFF` — distance between two dates

`DATE_DIFF` returns how far apart two dates are. Shape:

```sql
DATE_DIFF(later_date, earlier_date, unit)
```

The `unit` is `DAY`, `MONTH`, `YEAR`, etc.

How many days passed between Disneyland opening and Magic Kingdom opening?

In [ ]:
%%bigquery
SELECT
  DATE_DIFF(DATE '1971-10-01', DATE '1955-07-17', DAY) AS days_between
FROM disney_lessons.parks
LIMIT 1

(The `LIMIT 1` is just to avoid getting the same constant value 12 times — once per row in `parks`. There are cleaner ways, but `LIMIT 1` works fine here.)

## `CURRENT_DATE` — today

`CURRENT_DATE` returns today's date. Combine it with `DATE_DIFF` to get ages.

How old is each park, in years?

In [ ]:
%%bigquery
SELECT
  park_name,
  opened_date,
  DATE_DIFF(CURRENT_DATE, opened_date, YEAR) AS age_years
FROM disney_lessons.parks
ORDER BY age_years DESC

Disneyland comes out on top — it's been around the longest.

## Joins with `parks` and `attractions`

The `attractions` table also has 25 rows, one per famous ride/landmark:

| Column | Type | Meaning |
|--------|------|---------|
| `attraction_name` | text | Name of the attraction |
| `park_name` | text | Which park (matches `parks.park_name`) |
| `type` | text | "ride", "landmark", "area" |
| `opened_date` | DATE | When it opened |
| `min_height_inches` | number (nullable!) | Minimum rider height. NULL if no requirement. |

Take a look:

In [ ]:
%%bigquery
SELECT *
FROM disney_lessons.attractions
LIMIT 5

Same join pattern as Lesson 06, but now with date columns. Most recent attraction at each park:

In [ ]:
%%bigquery
SELECT
  p.park_name,
  MAX(a.opened_date) AS newest_attraction_date
FROM disney_lessons.parks AS p
INNER JOIN disney_lessons.attractions AS a
  ON a.park_name = p.park_name
GROUP BY p.park_name
ORDER BY newest_attraction_date DESC

## A quick note on `NULL` (revisited)

The `attractions` table has some `NULL` values in `min_height_inches` — landmarks and walk-through areas don't have height requirements.

Filter for attractions with **no** height requirement:

In [ ]:
%%bigquery
SELECT
  attraction_name,
  type,
  min_height_inches
FROM disney_lessons.attractions
WHERE min_height_inches IS NULL

And the opposite — attractions that *do* have a height requirement:

In [ ]:
%%bigquery
SELECT
  attraction_name,
  min_height_inches
FROM disney_lessons.attractions
WHERE min_height_inches IS NOT NULL
ORDER BY min_height_inches DESC

Remember: comparing to `NULL` with `=` doesn't work. **Always use `IS NULL` / `IS NOT NULL`.**

## Exercises

### Exercise 1

Find every **park opened in the 2000s** (between 2000 and 2009 inclusive). Show `park_name`, `location`, `opened_date`, sorted by date.

In [ ]:
# Your query here

### Exercise 2

Show each **decade** and how many **parks** opened in it. Use `EXTRACT(YEAR FROM ...)` and integer math (`year DIV 10 * 10` or similar) to bucket into decades — or use a `CASE` if that's easier.

(Tip: in GoogleSQL, `FLOOR(year / 10) * 10` gives you the decade.)

In [ ]:
# Your query here

### Exercise 3

For each **attraction**, show its name, its park, and its **age in years** today. Sort oldest first.

In [ ]:
# Your query here

### Exercise 4 (bonus)

What's the **average minimum height** across all attractions that **have** a height requirement? Round to 1 decimal.

In [ ]:
# Your query here

---

## Solutions

### Solution 1

In [ ]:
%%bigquery
SELECT
  park_name,
  location,
  opened_date
FROM disney_lessons.parks
WHERE EXTRACT(YEAR FROM opened_date) BETWEEN 2000 AND 2009
ORDER BY opened_date

### Solution 2

In [ ]:
%%bigquery
SELECT
  FLOOR(EXTRACT(YEAR FROM opened_date) / 10) * 10 AS decade,
  COUNT(*) AS parks_opened
FROM disney_lessons.parks
GROUP BY decade
ORDER BY decade

### Solution 3

In [ ]:
%%bigquery
SELECT
  attraction_name,
  park_name,
  DATE_DIFF(CURRENT_DATE, opened_date, YEAR) AS age_years
FROM disney_lessons.attractions
ORDER BY age_years DESC

### Solution 4

In [ ]:
%%bigquery
SELECT
  ROUND(AVG(min_height_inches), 1) AS avg_min_height_inches
FROM disney_lessons.attractions
WHERE min_height_inches IS NOT NULL

## What you've learned

- ✅ The `DATE` type and `DATE 'YYYY-MM-DD'` literals
- ✅ `EXTRACT(YEAR/MONTH/DAY FROM date)` to pull out parts
- ✅ `DATE_DIFF(later, earlier, unit)` to compute distance
- ✅ `CURRENT_DATE` for today
- ✅ `IS NULL` / `IS NOT NULL` for missing values

## Up next

You've now seen everything: filtering, sorting, aggregating, grouping, joining, conditional logic, and dates. Time to put it all together.

Open `09_capstone.ipynb`.